<a href="https://colab.research.google.com/github/LordLaalaPY/GitHub-Templates/blob/master/genesys_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import json
import time
import notebookutils
import PureCloudPlatformClientV2 as genesys
from PureCloudPlatformClientV2.rest import ApiException
from pyspark.sql import functions as F, Window
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    ArrayType, DoubleType,
)
var_lib = notebookutils.variableLibrary.getLibrary("Var_Client")
var_id = var_lib.getVariable("ID")
var_secret = var_lib.getVariable("Secret")
genesys.configuration.host = (
    genesys.PureCloudRegionHosts.eu_central_2.get_api_host()
)
apiclient = genesys.api_client.ApiClient().get_client_credentials_token(var_id,var_secret)
genesys.configuration.access_token = apiclient.access_token
auth_api = genesys.AuthorizationApi()

try:
    permissions = auth_api.get_authorization_permissions()
except ApiException as e:
    print(f"Fehler bei Anfrage: {e}")

if "apiclient" not in globals():
    raise RuntimeError(
        "Run your existing authentication cell first: "
        "apiclient = genesys.api_client.ApiClient().get_client_credentials_token(var_id, var_secret)"
    )

analytics = genesys.AnalyticsApi(apiclient)
spark.conf.set("spark.sql.session.timeZone", "UTC")

LOCAL_TZ = ZoneInfo("Europe/Zurich")
START = datetime(2026, 6, 1, tzinfo=LOCAL_TZ).astimezone(timezone.utc)
END_FIXED = datetime(2026, 10, 21, tzinfo=LOCAL_TZ).astimezone(timezone.utc)
END = min(END_FIXED, datetime.now(timezone.utc))
if END <= START:
    raise ValueError("End timestamp must be after June 1, 2026.")

# No voice filter: voice + all other conversation media types.
# Change to True only for a VOICE-ONLY test; all other channels will then be absent.
VOICE_ONLY = False
PAGE_SIZE_ASYNC = 1000
PAGE_SIZE_SYNC = 100
WRITE_BATCH = 500
ASYNC_WINDOW_DAYS = 7
SYNC_WINDOW_HOURS = 24
POLL_SECONDS = 5
JOB_TIMEOUT_SECONDS = 45 * 60

RAW_STAGE = "bronze.genesys_extract_stage_20260601_20261020_2"
RAW_TABLE = "bronze.genesys_conversations_corrected_20260601_20261020_2"
METRICS_TABLE = "silver.genesys_metrics_corrected_20260601_20261020_2"
SEGMENTS_TABLE = "silver.genesys_segments_corrected_20260601_20261020_2"
SESSIONS_TABLE = "silver.genesys_agent_sessions_corrected_20260601_20261020_2"

REF_USER_ID = "70922543-c022-4523-ad90-ec8f77183183"
REF_CONVERSATION_IDS = [
    "4c5b8bf6-703d-4807-bc79-972824ec0ba6",
    "78a68e28-6a1f-4f6a-8483-ccb8572f0177",
    "7f01154b-cf79-4fce-a179-c100708ecc2e",
]


def iso(dt):
    return dt.astimezone(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")


def native(response):
    return apiclient.sanitize_for_serialization(response)


def parse_datetime(value):
    if not value:
        return None
    d = value if isinstance(value, datetime) else datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    return d.replace(tzinfo=timezone.utc) if d.tzinfo is None else d.astimezone(timezone.utc)


def call_with_retry(fn):
    for attempt in range(7):
        try:
            return fn()
        except genesys.rest.ApiException as e:
            if getattr(e, "status", None) not in (429, 500, 502, 503, 504) or attempt == 6:
                raise
            pause = min(2 ** (attempt + 1), 60)
            print(f"API HTTP {e.status}: retry after {pause}s")
            time.sleep(pause)


def windows(start, end, step):
    cursor = start
    while cursor < end:
        following = min(end, cursor + step)
        yield cursor, following
        cursor = following


def voice_segment_filter():
    pred = genesys.SegmentDetailQueryPredicate()
    pred.type = "dimension"
    pred.dimension = "mediaType"
    pred.value = "voice"
    pred.operator = "matches"
    grouping = genesys.SegmentDetailQueryFilter()
    grouping.type = "and"
    grouping.predicates = [pred]
    return [grouping]


print("UTC window:", iso(START), "->", iso(END), "(exclusive)")
print("All media types:", not VOICE_ONLY)


def extract_async(start, end):
    query = genesys.AsyncConversationQuery()
    query.interval = f"{iso(start)}/{iso(end)}"
    # Leave start_of_day_interval_matching unset: include overlap;
    # filter conversationStart only ONCE after all pages/windows are acquired.
    if VOICE_ONLY:
        query.segment_filters = voice_segment_filter()

    submission = call_with_retry(lambda: analytics.post_analytics_conversations_details_jobs(query))
    job_id = submission.job_id
    if not job_id:
        raise RuntimeError("No job_id returned by Genesys")

    deadline = time.monotonic() + JOB_TIMEOUT_SECONDS
    while True:
        status = call_with_retry(lambda: analytics.get_analytics_conversations_details_job(job_id))
        state = (getattr(status, "state", "") or "").upper()
        if state == "FULFILLED":
            break
        if state in ("FAILED", "CANCELLED", "EXPIRED"):
            raise RuntimeError(f"Async job {job_id}: {native(status)}")
        if time.monotonic() > deadline:
            raise TimeoutError(f"Job {job_id} did not finish within {JOB_TIMEOUT_SECONDS}s")
        time.sleep(POLL_SECONDS)

    cursor = None
    seen_cursors = set()
    while True:
        args = {"page_size": PAGE_SIZE_ASYNC}
        if cursor:
            args["cursor"] = cursor
        response = call_with_retry(lambda: analytics.get_analytics_conversations_details_job_results(job_id, **args))
        for conversation in response.conversations or []:
            yield native(conversation)
        next_cursor = response.cursor
        if not next_cursor:
            break
        if next_cursor == cursor or next_cursor in seen_cursors:
            raise RuntimeError(f"Repeated cursor for async job {job_id}")
        seen_cursors.add(next_cursor)
        cursor = next_cursor


def query_sync_first(start, end):
    paging = genesys.PagingSpec()
    paging.page_size = PAGE_SIZE_SYNC
    paging.page_number = 1
    query = genesys.ConversationQuery()
    query.interval = f"{iso(start)}/{iso(end)}"
    query.paging = paging
    query.order = "asc"
    query.order_by = "conversationStart"
    if VOICE_ONLY:
        query.segment_filters = voice_segment_filter()
    result = call_with_retry(lambda: analytics.post_analytics_conversations_details_query(query))
    return query, result


def extract_sync(start, end):
    # Queries have a 100,000-record cap. Split rather than silently truncate.
    query, first = query_sync_first(start, end)
    hits = getattr(first, "total_hits", None)
    if hits is not None and hits >= 100000:
        if (end - start) <= timedelta(minutes=1):
            raise RuntimeError("Sync query >=100000 hits even within 1 minute; cannot guarantee completeness")
        middle = start + (end - start) / 2
        yield from extract_sync(start, middle)
        yield from extract_sync(middle, end)
        return

    current = first
    page_no = 1
    while True:
        conversations = current.conversations or []
        for conversation in conversations:
            yield native(conversation)
        if not conversations or len(conversations) < PAGE_SIZE_SYNC:
            break
        if hits is not None and page_no * PAGE_SIZE_SYNC >= hits:
            break
        if page_no >= 1000:
            # Avoid silently hitting the sync endpoint's 100k cap.
            raise RuntimeError(f"Sync query reached 100,000 records for interval {iso(start)}/{iso(end)}")
        page_no += 1
        query.paging.page_number = page_no
        current = call_with_retry(lambda: analytics.post_analytics_conversations_details_query(query))


In [ ]:
# Preflight happens BEFORE the staging table is overwritten.
for name in ["extract_async", "query_sync_first", "extract_sync"]:
    if not callable(globals().get(name)):
        raise RuntimeError(
            f"{name} fehlt. Zuerst die gesamte Vorbereitungszelle "
            "'1 — Einrichtung, historische und aktuelle API-Funktionen' ausfuehren. "
            "Dadurch wird verhindert, dass die Staging-Tabelle versehentlich geleert wird."
        )

try:
    availability = call_with_retry(lambda: analytics.get_analytics_conversations_details_jobs_availability())
    archive_end = parse_datetime(availability.data_availability_date)
except Exception as e:
    raise RuntimeError(
        "Could not determine async data availability. Aborting instead of silently missing dates. "
        "Check permissions for /analytics/conversations/details/jobs/availability."
    ) from e
if archive_end is None:
    raise RuntimeError("Genesys returned no dataAvailabilityDate. Cannot guarantee complete historical export.")
archive_end = max(START, min(END, archive_end))
recent_start = max(START, archive_end - timedelta(days=1))  # 1-day intentional overlap

print("Historical async endpoint through:", iso(archive_end))
print("Recent synchronous endpoint from:", iso(recent_start))

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

raw_schema = StructType([
    StructField("conversationId", StringType(), True),
    StructField("conversationStart", StringType(), True),
    StructField("conversationEnd", StringType(), True),
    StructField("rawJson", StringType(), False),
    StructField("_source", StringType(), False),
    StructField("_windowStart", StringType(), False),
    StructField("_windowEnd", StringType(), False),
    StructField("_retrievedAt", StringType(), False),
    StructField("_jsonLength", LongType(), False),
])

# Fresh, isolated test output every run; original 'bronze.genesys_voice_test_20260601' is untouched.
spark.createDataFrame([], raw_schema).write.format("delta").mode("overwrite").saveAsTable(RAW_STAGE)


def persist_window(fetcher, start, end, source):
    records = []
    total = 0
    for c in fetcher(start, end):
        cid = c.get("conversationId")
        if not cid:
            raise RuntimeError("Missing conversationId in returned conversation; cannot deduplicate safely")
        raw = json.dumps(c, ensure_ascii=False, default=str, separators=(",", ":"))
        records.append({
            "conversationId": cid,
            "conversationStart": c.get("conversationStart"),
            "conversationEnd": c.get("conversationEnd"),
            "rawJson": raw,
            "_source": source,
            "_windowStart": iso(start),
            "_windowEnd": iso(end),
            "_retrievedAt": iso(datetime.now(timezone.utc)),
            "_jsonLength": len(raw),
        })
        total += 1
        if len(records) >= WRITE_BATCH:
            spark.createDataFrame(records, raw_schema).write.format("delta").mode("append").saveAsTable(RAW_STAGE)
            records.clear()
    if records:
        spark.createDataFrame(records, raw_schema).write.format("delta").mode("append").saveAsTable(RAW_STAGE)
    print(f"{source:5} {iso(start)} -> {iso(end)} : {total} returned records")
    return total


window_log = []
for start, end in windows(START, archive_end, timedelta(days=ASYNC_WINDOW_DAYS)):
    count = persist_window(extract_async, start, end, "async")
    window_log.append((iso(start), iso(end), "async", count))
for start, end in windows(recent_start, END, timedelta(hours=SYNC_WINDOW_HOURS)):
    count = persist_window(extract_sync, start, end, "sync")
    window_log.append((iso(start), iso(end), "sync", count))

print("Window counts:")
display(spark.createDataFrame(window_log, ["windowStart", "windowEnd", "source", "returnedRecords"]))

In [ ]:
stage = spark.table(RAW_STAGE)
raw_with_ts = (
    stage
    .withColumn("_startUtc", F.to_timestamp("conversationStart"))
    .withColumn("_endUtc", F.to_timestamp("conversationEnd"))
    .withColumn("_retrievedUtc", F.to_timestamp("_retrievedAt"))
)

# Prefer the most complete conversation (latest end, then richest payload).
rank = Window.partitionBy("conversationId").orderBy(
    F.col("_endUtc").desc_nulls_last(),
    F.col("_jsonLength").desc(),
    F.col("_retrievedUtc").desc_nulls_last(),
)

keep = (
    raw_with_ts
    .withColumn("_rowRank", F.row_number().over(rank))
    .filter(F.col("_rowRank") == 1)
    .filter(
        (F.col("_startUtc") >= F.lit(START)) &
        (F.col("_startUtc") < F.lit(END))
    )
    .drop("_rowRank", "_startUtc", "_endUtc", "_retrievedUtc")
)

# Materialize final output independently of its source table.
keep.write.format("delta").mode("overwrite").saveAsTable(RAW_TABLE)
print("Unique conversations in requested period:", spark.table(RAW_TABLE).count())

In [ ]:
# The full original nested JSON remains available in RAW_TABLE.
metric_schema = StructType([
    StructField("name", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("emitDate", StringType(), True),
])
segment_schema = StructType([
    StructField("segmentType", StringType(), True),
    StructField("segmentStart", StringType(), True),
    StructField("segmentEnd", StringType(), True),
    StructField("queueId", StringType(), True),
    StructField("wrapUpCode", StringType(), True),
    StructField("disconnectType", StringType(), True),
])
session_schema = StructType([
    StructField("sessionId", StringType(), True),
    StructField("mediaType", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("metrics", ArrayType(metric_schema), True),
    StructField("segments", ArrayType(segment_schema), True),
])
participant_schema = StructType([
    StructField("participantId", StringType(), True),
    StructField("userId", StringType(), True),
    StructField("purpose", StringType(), True),
    StructField("sessions", ArrayType(session_schema), True),
])
conversation_schema = StructType([
    StructField("conversationId", StringType(), True),
    StructField("conversationStart", StringType(), True),
    StructField("conversationEnd", StringType(), True),
    StructField("originatingDirection", StringType(), True),
    StructField("participants", ArrayType(participant_schema), True),
])

parsed = spark.table(RAW_TABLE).select(F.from_json("rawJson", conversation_schema).alias("c"))
participants = parsed.select("c.conversationId", "c.conversationStart", "c.conversationEnd",
                             "c.originatingDirection", F.explode_outer("c.participants").alias("p"))
sessions = (
    participants.select("conversationId", "conversationStart", "conversationEnd", "originatingDirection",
                        F.col("p.participantId").alias("participantId"),
                        F.col("p.userId").alias("userId"),
                        F.col("p.purpose").alias("purpose"),
                        F.explode_outer("p.sessions").alias("s"))
    .select("conversationId", "conversationStart", "conversationEnd", "originatingDirection",
            "participantId", "userId", "purpose",
            F.col("s.sessionId").alias("sessionId"),
            F.col("s.mediaType").alias("mediaType"),
            F.col("s.direction").alias("direction"),
            F.col("s.metrics").alias("metrics"),
            F.col("s.segments").alias("segments"))
    .filter(F.col("sessionId").isNotNull())
)

ids = ["conversationId", "conversationStart", "conversationEnd", "originatingDirection",
       "participantId", "userId", "purpose", "sessionId", "mediaType", "direction"]

all_metrics = (
    sessions.select(*ids, F.posexplode_outer("metrics").alias("metricOrdinal", "metric"))
    .filter(F.col("metric").isNotNull())
    .select(*ids, "metricOrdinal", F.col("metric.name").alias("name"),
            F.col("metric.value").alias("value"), F.col("metric.emitDate").alias("emitDate"))
)
all_metrics.write.format("delta").mode("overwrite").saveAsTable(METRICS_TABLE)

all_segments = (
    sessions.select(*ids, F.posexplode_outer("segments").alias("segmentOrdinal", "segment"))
    .filter(F.col("segment").isNotNull())
    .select(*ids, "segmentOrdinal", "segment.*")
)
all_segments.write.format("delta").mode("overwrite").saveAsTable(SEGMENTS_TABLE)

In [ ]:
# Keep Genesys metric names in milliseconds and add explicitly named seconds.
metric_names = [
    "tTalk", "tTalkComplete", "tAcw", "tHeld", "tHeldComplete",
    "tHandle", "tDialing", "tContacting", "tAcd", "tIvr",
    "tAnswered", "tAlert", "tWait", "tConnected",
]
agent_metrics = spark.table(METRICS_TABLE).filter(F.col("purpose") == "agent")
key_cols = ["conversationId", "participantId", "userId", "sessionId", "mediaType"]

metric_sums = agent_metrics.groupBy(*key_cols).agg(
    *[F.sum(F.when(F.col("name") == name, F.col("value"))).alias(name) for name in metric_names],
    F.sum(F.when(F.col("name") == "tAcw", 1).otherwise(0)).alias("acwMetricCount"),
    F.sum(F.when(F.col("name") == "tHandle", 1).otherwise(0)).alias("handleMetricCount"),
    F.sum(F.when(F.col("name") == "tAnswered", 1).otherwise(0)).alias("answeredMetricCount"),
)

agent_sessions = (
    sessions.filter(F.col("purpose") == "agent")
    .select(*key_cols, "conversationStart", "conversationEnd", "originatingDirection", "direction")
    .join(metric_sums, key_cols, "left")
)
# Derived field for simple reporting only; never replaces native API fields.
# If both are present and disagree (e.g. after transfers), the conversation's
# original direction takes precedence. An absent direction stays NULL.
valid_values = ["inbound", "outbound"]
origin = F.lower(F.col("originatingDirection"))
session_dir = F.lower(F.col("direction"))
agent_sessions = agent_sessions.withColumn(
    "directionForReporting",
    F.when(origin.isin(*valid_values), origin)
     .when(session_dir.isin(*valid_values), session_dir)
     .otherwise(F.lit(None).cast("string"))
)
for name in metric_names:
    agent_sessions = agent_sessions.withColumn(name + "Seconds", F.col(name) / F.lit(1000.0))

# Diagnostics only: not replacing Genesys tHandle.
agent_sessions = agent_sessions.withColumn(
    "handleComponentsSeconds",
    F.coalesce(F.col("tTalkSeconds"), F.lit(0.0))
    + F.coalesce(F.col("tAcwSeconds"), F.lit(0.0))
    + F.coalesce(F.col("tHeldSeconds"), F.lit(0.0))
    + F.coalesce(F.col("tDialingSeconds"), F.lit(0.0))
    + F.coalesce(F.col("tContactingSeconds"), F.lit(0.0))
).withColumn(
    "handleDifferenceSeconds", F.col("tHandleSeconds") - F.col("handleComponentsSeconds")
)

agent_sessions.write.format("delta").mode("overwrite").saveAsTable(SESSIONS_TABLE)

In [ ]:
from pyspark.sql import functions as SF
reference = (
    spark.table(SESSIONS_TABLE)
    .filter((SF.col("userId") == REF_USER_ID) & (SF.col("mediaType") == "voice"))
    .filter(SF.col("conversationId").isin(REF_CONVERSATION_IDS))
)
print("Expected (three calls): Talk 89.808s | ACW 13.021s | Handle 115.962s")
display(reference.select(
    "conversationId", "conversationStart", "mediaType", "userId", "sessionId",
    "originatingDirection", "direction", "directionForReporting",
    "tTalkSeconds", "tAcwSeconds", "tHandleSeconds", "tHeldSeconds",
    "tDialingSeconds", "tContactingSeconds", "handleDifferenceSeconds"
).orderBy("conversationStart"))

check = reference.agg(
    SF.countDistinct("conversationId").alias("conversations"),
    SF.sum("tTalkSeconds").alias("talkSeconds"),
    SF.sum("tAcwSeconds").alias("acwSeconds"),
    SF.sum("tHandleSeconds").alias("handleSeconds"),
).first()
print("REFERENCE CHECK:", check.asDict())
expected = {"talkSeconds": 89.808, "acwSeconds": 13.021, "handleSeconds": 115.962}
if check["conversations"] != 3 or any(
    check[k] is None or abs(check[k] - v) > 0.1 for k, v in expected.items()
):
    raise RuntimeError(
        "REFERENCE MISMATCH: 3 reference conversations missing or metrics differ. "
        "Investigate permissions, job completion, and API returned fields BEFORE using the export. "
        f"Observed: {check.asDict()}"
    )
print("PASS: known Sept 4 reference metrics match Genesys UI.")
# Expected: outbound 10:39, inbound 10:35 and 10:32. A mismatch is shown
# for investigation rather than silently modifying any native field.
expected_direction = {
    "4c5b8bf6-703d-4807-bc79-972824ec0ba6": "outbound",
    "78a68e28-6a1f-4f6a-8483-ccb8572f0177": "inbound",
    "7f01154b-cf79-4fce-a179-c100708ecc2e": "inbound",
}
print("Direction check (compare with Genesys Interactions):")
for row in reference.select("conversationId", "originatingDirection", "direction", "directionForReporting").distinct().collect():
    expected_value = expected_direction.get(row["conversationId"])
    actual = row["directionForReporting"]
    print(row["conversationId"], "expected:", expected_value,
          "originatingDirection:", row["originatingDirection"],
          "direction:", row["direction"],
          "directionForReporting:", actual,
          "MATCH" if actual == expected_value else "REVIEW")

In [ ]:
print("Saved tables:", RAW_TABLE, METRICS_TABLE, SEGMENTS_TABLE, SESSIONS_TABLE, sep="\n - ")
print("Conversations:", spark.table(RAW_TABLE).count())
print("Agent sessions:", spark.table(SESSIONS_TABLE).count())
display(
    spark.table(SESSIONS_TABLE)
    .select("conversationId", "conversationStart", "userId", "mediaType",
            "originatingDirection", "direction", "directionForReporting",
            "tTalkSeconds", "tAcwSeconds", "tHeldSeconds", "tHandleSeconds",
            "handleDifferenceSeconds")
    .orderBy(F.desc("conversationStart"))
    .limit(100)
)
print("Agent session counts by media type:")
display(spark.table(SESSIONS_TABLE).groupBy("mediaType").count().orderBy(F.desc("count")))
print("Direction counts by media type (agent sessions, not unique conversations):")
display(spark.table(SESSIONS_TABLE).groupBy("mediaType", "directionForReporting").count().orderBy("mediaType", "directionForReporting"))
